# 01 — Data Exploration

Explore the raw data pipeline: load prices, check data quality, visualise
correlations, and get a Claude interpretation of the feature landscape.

**Cost per full run**: ~£0.02 (one Claude API call at ~1k tokens).

The notebook runs fully without an API key — charts and metrics render
normally; the interpretation cell shows "Interpretation unavailable".

### Path setup

We insert the project root so that `src` is importable from the
`notebooks/` directory.  This is the standard pattern for Jupyter
notebooks that live one level below the project root.

In [ ]:
import sys
sys.path.insert(0, "..")

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.utils.config import load_config
from src.services.data_service import DataService
from src.notebooks.formatters import format_feature_stats, display_interpretation
from src.notebooks.research_log import ResearchLog

In [ ]:
config = load_config()
data_service = DataService(config)
research_log = ResearchLog(config)

tickers = config["universe"]["tickers"]
print(f"Universe: {tickers}")

## Section 1 — Load Data

In [ ]:
prices = data_service.get_prices()
print(f"Prices shape: {prices.shape}")
print(f"Date range: {prices.index.min()} -> {prices.index.max()}")
prices.tail()

## Section 2 — Data Quality

In [ ]:
# Missing values per column.
missing = prices.isna().sum()
missing_pct = (prices.isna().mean() * 100).round(2)

quality_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
print("Data quality summary:")
quality_df

In [ ]:
# Distribution of daily returns.
returns = prices.pct_change().dropna()

fig = px.histogram(
    returns.melt(var_name="Ticker", value_name="Return"),
    x="Return",
    color="Ticker",
    nbins=80,
    title="Distribution of Daily Returns",
    barmode="overlay",
    opacity=0.6,
)
fig.show()

In [ ]:
# Detect outliers (returns beyond 3 std deviations).
z_scores = (returns - returns.mean()) / returns.std()
outliers = (z_scores.abs() > 3).sum()
print("Outlier counts (|z| > 3):")
print(outliers)

## Section 3 — Correlation Heatmap

In [ ]:
corr_matrix = returns.corr()

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    title="Pairwise Return Correlations",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
)
fig.show()

## Section 4 — Claude Interpretation

In [ ]:
# Load features for the first ticker to analyse.
ticker = tickers[0]
try:
    features_df = data_service.get_features(ticker)
    feature_stats = format_feature_stats(features_df)
except Exception as e:
    print(f"Could not load features for {ticker}: {e}")
    feature_stats = {"feature_count": 0, "missing_pct": {}, "autocorrelation": {}, "target_correlations": []}

# Add data quality summary.
feature_stats["data_quality"] = {
    "total_rows": len(prices),
    "date_range": f"{prices.index.min()} -> {prices.index.max()}",
    "missing_pct": missing_pct.to_dict(),
    "outlier_counts": outliers.to_dict(),
    "mean_correlation": round(float(corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)].mean()), 4),
}

print("Data prepared for interpretation:")
print(f"  Features: {feature_stats.get('feature_count', 0)}")
print(f"  Rows: {feature_stats['data_quality']['total_rows']}")

In [ ]:
try:
    from src.notebooks.claude_interpreter import QuantInterpreter

    interpreter = QuantInterpreter(config)
    interpretation = interpreter.interpret("feature_analysis", feature_stats)
except (EnvironmentError, ImportError) as e:
    print(f"Claude interpretation unavailable: {e}")
    interpretation = {
        "summary": "Interpretation unavailable (no API key set)",
        "observations": [],
        "warnings": ["Set ANTHROPIC_API_KEY to enable interpretations"],
        "suggestions": [],
        "confidence": "low",
    }

display_interpretation(interpretation)

In [ ]:
# Log the research entry.
research_log.log_entry(
    notebook="01_data_exploration",
    task="feature_analysis",
    data_summary=feature_stats,
    interpretation=interpretation,
)
print("Entry logged.")